In [156]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
from ehrpredict import EHRPredict
import polars as pl
import joblib
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')

In [158]:
data = {
    'demo':pl.read_parquet('/home/niels/Documents/better_thunder/mgb/demo.parquet').select(
        pl.col('bdsp_patient_id').cast(pl.String).alias('id'),
        pl.col('sex'),
        pl.col('date_of_birth').cast(pl.Date)),
    'icd':pl.read_parquet('/home/niels/Documents/better_thunder/mgb/icd.parquet').select(
        pl.col('bdsp_patient_id').cast(pl.String).alias('id'),
        pl.col('icd'),
        pl.col('date_icd').cast(pl.Date).alias('date')
    ),
    'med':pl.read_parquet('/home/niels/Documents/better_thunder/mgb/med.parquet').select(
        pl.col('bdsp_patient_id').cast(pl.String).alias('id'),
        pl.col('med'),
        pl.col('date_start_med').str.strptime(pl.Date, '%Y-%m-%d %H:%M:%S%.f').alias('date')
    ),
    'note':pl.read_parquet('/home/niels/Documents/better_thunder/mgb/note/1996.parquet').select(
        pl.col('bdsp_patient_id').cast(pl.String).alias('id'),
        pl.col('note'),
        pl.col('date_note').cast(pl.Date).alias('date')
    )
}

In [162]:
predictor = EHRPredict()

feat, pred = predictor.predict(
    'epilepsy',
    data
)

INFO:root:Loading model from models/epilepsy/epilepsy_model.joblib
INFO:root:Preprocessing started at 2025-03-05 15:25:49.273660
INFO:root:Generating features for n = 27460
INFO:root:Demo preprocessing started at 2025-03-05 15:25:51.759991
INFO:root:Demo preprocessing finished at 2025-03-05 15:25:51.771425
INFO:root:ICD preprocessing started at 2025-03-05 15:25:51.772225
INFO:root:ICD preprocessing finished at 2025-03-05 15:25:51.933395
INFO:root:Med preprocessing started at 2025-03-05 15:25:51.934044
INFO:root:Med preprocessing finished at 2025-03-05 15:26:10.349914
INFO:root:Note preprocessing started at 2025-03-05 15:26:10.350800
2025-03-05 15:26:28,074	INFO worker.py:1786 -- Started a local Ray instance.


(pid=2318039) Processing notes 0:   0%|          | 0.00/27.5k [00:00<?, ?it/s]

INFO:root:Note preprocessing finished at 2025-03-05 15:27:08.151445
INFO:root:Preprocessing finished at 2025-03-05 15:27:08.167928
INFO:root:Prediction started at 2025-03-05 15:27:08.169736
INFO:root:Getting predictions
INFO:root:Prediction finished at 2025-03-05 15:27:08.242402


In [169]:
print(f'Total input: {len(pred)}')
print(f'Total unique patients: {len(pred["id"].unique())}')
print(f'Positive predictions: {len(pred.filter(pl.col("prediction") == 1))}')
print(f'Negative predictions: {len(pred.filter(pl.col("prediction") == 0))}')
print(f'Total patients with 1+ positive prediction: {len(pred.filter(pl.col("prediction") == 1)["id"].unique())}')

Total input: 27460
Total unique patients: 8513
Positive predictions: 859
Negative predictions: 26601
Total patients with 1+ positive prediction: 218
